In [1]:
import argparse
import json
from pathlib import Path
from typing import Iterable, List
from tqdm import tqdm
import pandas as pd
from sentence_transformers import SentenceTransformer

In [10]:
import copy

In [2]:
def read_csv_robust(path: Path, encodings: Iterable[str] = ("utf-8", "latin-1")) -> pd.DataFrame:
    last_err = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Failed to read CSV {path} ({last_err})")

def build_texts(df: pd.DataFrame, cols: List[str], joiner: str) -> pd.Series:
    parts = [df[c].fillna("").astype(str).str.strip() if c in df.columns
             else pd.Series([""] * len(df), index=df.index) for c in cols]
    s = parts[0]
    for p in parts[1:]:
        s = s + joiner + p
    # collapse multiple joiners caused by empties, and strip
    j = joiner.strip()
    if j:
        s = s.str.replace(rf"(?:\s*{j}\s*)+", f" {j} ", regex=True)
    return s.str.strip()


def process_one_file(
    model: SentenceTransformer,
    in_csv: Path,
    out_csv: Path,
    cols: List[str],
    joiner: str,
    batch_size: int,
    normalize: bool,
):
    df = read_csv_robust(in_csv)

    missing_cols = [c for c in cols if c not in df.columns]
    if missing_cols:
        print(f"[WARN] {in_csv.name}: missing columns {missing_cols}; they will be treated as empty strings.")

    sbert_text = build_texts(df, cols, joiner)

    embeddings = model.encode(
        sbert_text.tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=normalize,
    )

    out_df = df.copy()
    out_df["sbert_text"] = sbert_text
    out_df["embedding_json"] = [json.dumps(vec.tolist()) for vec in embeddings]

    out_csv.parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(out_csv, index=False)
    print(f"[OK] {in_csv.name} -> {out_csv.name} (rows: {len(out_df)})")

In [9]:
df_train = pd.read_csv(Path('/data/elugos/event_embedding/train_20251214.csv'))

/tmp/ipykernel_2643767/499715571.py:1: DtypeWarning: Columns (40,41) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv(Path('/data/elugos/event_embedding/train_20251214.csv'))


In [24]:
df_train.head()

,index,GlobalEventID,date,Year,Actor1Code,Actor1Name,Actor1CountryCode,Actor1EthnicCode,Actor2Code,Actor2Name,...,target,title,text,description,sbert_text_title,embedding_json_title,sbert_text,embedding_json,first_para,first_para_emb
0,0,1200203233,20240918,2024,USA,UNITED STATES,USA,NaN,CRM,GANG,...,bostonherald,Tren de Aragua gang started in Venezuela’s pri...,MIAMI (AP) — Former federal agent Was Tabor sa...,The gang has exploded into the presidential ca...,Tren de Aragua gang started in Venezuela’s pri...,"[-0.002818030072376132, -0.01742059737443924, ...",NaN,NaN,MIAMI (AP) — Former federal agent Was Tabor sa...,"[-0.08164525777101517, -0.027655666694045067, ..."
1,1,1200203234,20240918,2024,USA,FLORIDA,USA,NaN,GOV,ADMINISTRATION,...,bostonherald,Tren de Aragua gang started in Venezuela’s pri...,MIAMI (AP) — Former federal agent Was Tabor sa...,The gang has exploded into the presidential ca...,Tren de Aragua gang started in Venezuela’s pri...,"[-0.002818030072376132, -0.01742059737443924, ...",NaN,NaN,MIAMI (AP) — Former federal agent Was Tabor sa...,"[-0.08164525777101517, -0.027655666694045067, ..."
2,2,1200203284,20240925,2024,NaN,NaN,NaN,NaN,CHR,CHRISTIAN,...,brnow,"Yarnell: In times of competing allegiances, re...",NASHVILLE (BP) — The word “tumultuous” is one ...,NASHVILLE (BP) — The word “tumultuous” is one ...,"Yarnell: In times of competing allegiances, re...","[-0.0645490437746048, 0.05730828642845154, 0.0...",NaN,NaN,NASHVILLE (BP) — The word “tumultuous” is one ...,"[0.017953015863895416, -0.06336908042430878, -..."
3,3,1200203520,20240925,2024,NaN,NaN,NaN,NaN,USA,FLORIDA,...,yahoo,How Strong Could Hurricane Helene Get When It ...,Yahoo is using AI to generate takeaways from t...,Weather experts are expecting Hurricane Helene...,How Strong Could Hurricane Helene Get When It ...,"[-0.018027156591415405, 0.024802444502711296, ...",NaN,NaN,Yahoo is using AI to generate takeaways from t...,"[-0.018082909286022186, 0.06462761014699936, -..."
4,4,1200203521,20240925,2024,NaN,NaN,NaN,NaN,USA,FLORIDA,...,yahoo,How Strong Could Hurricane Helene Get When It ...,Yahoo is using AI to generate takeaways from t...,Weather experts are expecting Hurricane Helene...,How Strong Could Hurricane Helene Get When It ...,"[-0.018027156591415405, 0.024802444502711296, ...",NaN,NaN,Yahoo is using AI to generate takeaways from t...,"[-0.018082909286022186, 0.06462761014699936, -..."


In [12]:
df_txt= df_train.drop_duplicates(subset=['text'])
df_gid =df_train.drop_duplicates(subset=['GlobalEventID'])

In [ ]:
df_title = df_train.drop_duplicates()

In [23]:
df_txt

,index,GlobalEventID,date,Year,Actor1Code,Actor1Name,Actor1CountryCode,Actor1EthnicCode,Actor2Code,Actor2Name,...,target,title,text,description,sbert_text_title,embedding_json_title,sbert_text,embedding_json,first_para,first_para_emb
0,0,1200203233,20240918,2024,USA,UNITED STATES,USA,NaN,CRM,GANG,...,bostonherald,Tren de Aragua gang started in Venezuela’s pri...,MIAMI (AP) — Former federal agent Was Tabor sa...,The gang has exploded into the presidential ca...,Tren de Aragua gang started in Venezuela’s pri...,"[-0.002818030072376132, -0.01742059737443924, ...",NaN,NaN,MIAMI (AP) — Former federal agent Was Tabor sa...,"[-0.08164525777101517, -0.027655666694045067, ..."
2,2,1200203284,20240925,2024,NaN,NaN,NaN,NaN,CHR,CHRISTIAN,...,brnow,"Yarnell: In times of competing allegiances, re...",NASHVILLE (BP) — The word “tumultuous” is one ...,NASHVILLE (BP) — The word “tumultuous” is one ...,"Yarnell: In times of competing allegiances, re...","[-0.0645490437746048, 0.05730828642845154, 0.0...",NaN,NaN,NASHVILLE (BP) — The word “tumultuous” is one ...,"[0.017953015863895416, -0.06336908042430878, -..."
3,3,1200203520,20240925,2024,NaN,NaN,NaN,NaN,USA,FLORIDA,...,yahoo,How Strong Could Hurricane Helene Get When It ...,Yahoo is using AI to generate takeaways from t...,Weather experts are expecting Hurricane Helene...,How Strong Could Hurricane Helene Get When It ...,"[-0.018027156591415405, 0.024802444502711296, ...",NaN,NaN,Yahoo is using AI to generate takeaways from t...,"[-0.018082909286022186, 0.06462761014699936, -..."
5,5,1200203590,20240925,2024,NaN,NaN,NaN,NaN,USA,FLORIDA,...,yahoo,"Florida, Georgia Teamsters break with national...",Yahoo is using AI to generate takeaways from t...,Since the non-endorsement by the national gove...,"Florida, Georgia Teamsters break with national...","[-0.049125298857688904, -0.03338903561234474, ...",NaN,NaN,Yahoo is using AI to generate takeaways from t...,"[-0.018082909286022186, 0.06462761014699936, -..."
9,9,1200204213,20240925,2024,EDU,STUDENT,NaN,NaN,NaN,NaN,...,news,A day to remember: FIU celebrates Top 50,The university that never stops has experience...,Read how members of the university community r...,A day to remember: FIU celebrates Top 50,"[0.021655291318893433, -0.011736852116882801, ...",NaN,NaN,The university that never stops has experience...,"[0.07788405567407608, -0.07917836308479309, 0...."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118442,122969,679730555,20170808,2017,RUS,MOSCOW,RUS,NaN,USA,UNITED STATES,...,centralmaine,New superintendent in Anson-based RSU 74 sees ...,NORTH ANSON — Michael Tracy begins work as the...,"Michael Tracy, who took over July 1, said poss...",New superintendent in Anson-based RSU 74 sees ...,"[0.0074782222509384155, -0.06746511161327362, ...",NaN,NaN,NORTH ANSON — Michael Tracy begins work as the...,"[-0.023405712097883224, 0.002474960871040821, ..."
118443,122970,679730864,20170808,2017,USA,UNITED STATES,USA,NaN,GOV,PRESIDENT,...,newsobserver,"Trump, Cooper hold strong positions on Atlanti...",President Donald Trump promised an America-Fir...,President Donald Trump promised an America-Fir...,"Trump, Cooper hold strong positions on Atlanti...","[0.001870636478997767, -0.035346224904060364, ...",NaN,NaN,President Donald Trump promised an America-Fir...,"[-0.07812009751796722, 0.0563594251871109, 0.0..."
118446,122973,679732457,20170808,2017,EDU,SCHOOL,NaN,NaN,NaN,NaN,...,newswise,WVU Providing Multi-Pronged Approach Solving t...,President Donald Trump was briefed today about...,President Donald Trump was briefed today about...,WVU Providing Multi-Pronged Approach Solving t...,"[0.015047898516058922, -0.004980273544788361, ...",NaN,NaN,President Donald Trump was briefed today about...,"[0.0483584962785244, -0.012323571369051933, 0...."
118448,122977,679733469,20170808,2017,USA,UNITED STATES,USA,NaN,NaN,NaN,...,washingtonpost,Opinion | Bricks and mortar may be key in the ...,Ed Gillespie and Ralph Northam

In [15]:
from transformers import AutoTokenizer, BartForConditionalGeneration

model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")

ARTICLE_TO_SUMMARIZE = (
    "PG&E stated it scheduled the blackouts in response to forecasts for high winds "
    "amid dry conditions. The aim is to reduce the risk of wildfires. Nearly 800 thousand customers were "
    "scheduled to be affected by the shutoffs which were expected to last through at least midday tomorrow."
)
inputs = tokenizer([ARTICLE_TO_SUMMARIZE], max_length=1024, return_tensors="pt")

# Generate Summary
summary_ids = model.generate(inputs["input_ids"], num_beams=2, min_length=0, max_length=20)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


In [20]:
from transformers import AutoTokenizer, pipeline

# Load BART (CPU-safe)
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)

summarizer = pipeline(
    "summarization",
    model=model_name,
    tokenizer=model_name,
    device="cpu"  # avoid GPU issues
)

###########################################################
# Function: split into <=900 token chunks (safe for BART)
###########################################################
def chunk_text(text, max_tokens=1000):
    tokens = tokenizer.encode(text)
    chunks = []
    
    for i in range(0, len(tokens), max_tokens):
        chunk = tokens[i : i + max_tokens]
        chunks.append(chunk)
    
    return chunks

Device set to use cpu


In [21]:
###########################################################
# Function: summarize long texts safely
###########################################################
def summarize(text):
    chunks = chunk_text(text)

    partial_summaries = []
    for token_chunk in chunks:
        chunk_text_decoded = tokenizer.decode(token_chunk, skip_special_tokens=True)

        summary = summarizer(
            chunk_text_decoded,
            max_length=150,
            min_length=60,
            do_sample=False
        )[0]["summary_text"]

        partial_summaries.append(summary)

    # If many partial summaries exist → summarize them too
    if len(partial_summaries) > 1:
        combined = " ".join(partial_summaries)
        final = summarizer(
            combined,
            max_length=150,
            min_length=60,
            do_sample=False
        )[0]["summary_text"]
        return final
    else:
        return partial_summaries[0]





def summarize_text_safe(text):
    if not isinstance(text, str) or not text.strip():
        return None
    return summarize(text)






In [22]:

tqdm.pandas()

df_txt["text_bart"] = df_txt["text"].progress_apply(summarize_text_safe)

print("Finished summarizing all patents!")

  0%|                                          | 8/47058 [01:09<98:44:02,  7.55s/it]Your max_length is set to 150, but your input_length is only 56. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=28)
Your max_length is set to 150, but your input_length is only 134. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=67)
  0%|                                        | 31/47058 [04:10<157:44:25, 12.08s/it]Your max_length is set to 150, but your input_length is only 50. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=25)
Your max_length is set to 150, but your input_length is only 142. Since this is a su

IndexError: index out of range in self